# AEGIS-ZERO — Interaktywny Demo Firewalla

Notebook pozwala na ręczne definiowanie reguł dostępu oraz testowanie pakietów przez dwuwarstwowy potok filtracji:

```
Pakiet ──► [Layer 1: Bloom Filter] ──► [Layer 2: Cuckoo Hash] ──► ALLOW / DROP
                  (BRAM, O(1))               (RAM, O(1))
```

**Kolejność komórek:** uruchamiaj od góry do dołu.

In [2]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), 'src'))

from common import FiveTuple, ip_to_int
from aegis_pipeline import AegisPipeline, Verdict

print("Moduły załadowane pomyślnie.")

Moduły załadowane pomyślnie.


---
## 1. Inicjalizacja firewalla

Tworzymy pustą instancję potoku. Na tym etapie żadna reguła nie jest jeszcze załadowana.

In [3]:
fw = AegisPipeline()

print(f"Firewall zainicjalizowany.")
print(f"  Bloom fill ratio  : {fw.bloom.fill_ratio:.2%}")
print(f"  Cuckoo load factor: {fw.cuckoo.load_factor:.2%}")

Firewall zainicjalizowany.
  Bloom fill ratio  : 0.00%
  Cuckoo load factor: 0.00%


---
## 2. Definiowanie reguł (whitelist)

Każda reguła to 5-krotka: `(src_ip, dst_ip, src_port, dst_port, proto)`.

Edytuj listę `MY_RULES` — możesz dodać dowolną liczbę wpisów.

In [4]:
# ── Edytuj tutaj ─────────────────────────────────────────────────────────────
MY_RULES = [
    # (src_ip,         dst_ip,           src_port, dst_port, proto)
    ("10.0.0.1",  "192.168.100.1",  50000,    443,      6),   # TCP/HTTPS
    ("10.0.0.2",  "192.168.100.1",  50001,    443,      6),
    ("10.0.0.3",  "192.168.100.1",  50002,    443,      6),
    ("10.0.1.10", "192.168.100.1",  60000,    443,      6),
    ("10.0.1.11", "192.168.100.1",  60001,    443,      6),
]
# ─────────────────────────────────────────────────────────────────────────────

rules = [
    FiveTuple(
        src_ip   = ip_to_int(r[0]),
        dst_ip   = ip_to_int(r[1]),
        src_port = r[2],
        dst_port = r[3],
        proto    = r[4],
    )
    for r in MY_RULES
]

fw.load_rules(rules)

print(f"Załadowano {len(rules)} reguł do firewalla.")
print(f"  Bloom fill ratio  : {fw.bloom.fill_ratio:.2%}")
print(f"  Cuckoo load factor: {fw.cuckoo.load_factor:.2%}")
print()
print("Aktywne reguły:")
print(f"  {'src_ip':<15} {'dst_ip':<15} {'sport':>6} {'dport':>6} {'proto':>5}")
print("  " + "-"*52)
for r in MY_RULES:
    print(f"  {r[0]:<15} {r[1]:<15} {r[2]:>6} {r[3]:>6} {r[4]:>5}")

Załadowano 5 reguł do firewalla.
  Bloom fill ratio  : 0.03%
  Cuckoo load factor: 0.01%

Aktywne reguły:
  src_ip          dst_ip           sport  dport proto
  ----------------------------------------------------
  10.0.0.1        192.168.100.1    50000    443     6
  10.0.0.2        192.168.100.1    50001    443     6
  10.0.0.3        192.168.100.1    50002    443     6
  10.0.1.10       192.168.100.1    60000    443     6
  10.0.1.11       192.168.100.1    60001    443     6


---
## 3. Wysyłanie pojedynczego pakietu

Zdefiniuj pakiet i sprawdź czy firewall go przepuści.

In [5]:
# ── Edytuj tutaj ─────────────────────────────────────────────────────────────
PACKET = {
    "src_ip"  : "10.0.0.1",       # zmień na dowolny IP
    "dst_ip"  : "192.168.100.1",
    "src_port": 50000,
    "dst_port": 443,
    "proto"   : 6,
}
# ─────────────────────────────────────────────────────────────────────────────

pkt = FiveTuple(
    src_ip   = ip_to_int(PACKET["src_ip"]),
    dst_ip   = ip_to_int(PACKET["dst_ip"]),
    src_port = PACKET["src_port"],
    dst_port = PACKET["dst_port"],
    proto    = PACKET["proto"],
)

verdict = fw.process(pkt)

SYMBOLS = {
    Verdict.ALLOW:   ("✅", "ALLOW  ", "Pakiet autoryzowany — wpuszczony do Cytadeli."),
    Verdict.DROP_L1: ("🛑", "DROP L1", "Odrzucony przez Bloom Filter (pre-filtr BRAM)."),
    Verdict.DROP_L2: ("⚠️ ", "DROP L2", "Fałszywy alarm L1 — odrzucony przez Cuckoo Hash."),
}

icon, label, desc = SYMBOLS[verdict]

print(f"Pakiet:  {PACKET['src_ip']}:{PACKET['src_port']} → {PACKET['dst_ip']}:{PACKET['dst_port']}  proto={PACKET['proto']}")
print(f"Werdykt: {icon}  {label}")
print(f"         {desc}")

Pakiet:  10.0.0.1:50000 → 192.168.100.1:443  proto=6
Werdykt: ✅  ALLOW  
         Pakiet autoryzowany — wpuszczony do Cytadeli.


---
## 4. Wysyłanie wielu pakietów naraz

Zdefiniuj listę pakietów testowych — każdy z nich przejdzie przez cały potok.

In [6]:
# ── Edytuj tutaj ─────────────────────────────────────────────────────────────
TEST_PACKETS = [
    # Autoryzowane
    ("10.0.0.1",   "192.168.100.1", 50000, 443, 6),
    ("10.0.0.2",   "192.168.100.1", 50001, 443, 6),
    ("10.0.1.10",  "192.168.100.1", 60000, 443, 6),
    # Nieautoryzowane IP
    ("172.16.0.1",  "192.168.100.1", 50000, 443, 6),
    ("192.168.1.5", "192.168.100.1", 12345, 443, 6),
    # Zły port
    ("10.0.0.1",   "192.168.100.1", 50000, 22,  6),
    # Zły protokół
    ("10.0.0.1",   "192.168.100.1", 50000, 443, 17)
]
# ─────────────────────────────────────────────────────────────────────────────

ICONS = {Verdict.ALLOW: "✅", Verdict.DROP_L1: "🛑", Verdict.DROP_L2: "⚠️ "}
LABELS = {Verdict.ALLOW: "ALLOW  ", Verdict.DROP_L1: "DROP L1", Verdict.DROP_L2: "DROP L2"}

from collections import Counter
stats: Counter = Counter()

print(f"{'src_ip':<15} {'dst_ip':<15} {'sp':>6} {'dp':>6} {'pr':>3}   {'werdykt'}")
print("-" * 70)

for raw in TEST_PACKETS:
    pkt = FiveTuple(
        src_ip=ip_to_int(raw[0]), dst_ip=ip_to_int(raw[1]),
        src_port=raw[2], dst_port=raw[3], proto=raw[4],
    )
    v = fw.process(pkt)
    stats[v] += 1
    print(f"{raw[0]:<15} {raw[1]:<15} {raw[2]:>6} {raw[3]:>6} {raw[4]:>3}   {ICONS[v]} {LABELS[v]}")

print("-" * 70)
print(f"Łącznie: {len(TEST_PACKETS)} pakietów")
print(f"  ✅  ALLOW  : {stats[Verdict.ALLOW]}")
print(f"  🛑  DROP L1: {stats[Verdict.DROP_L1]}")
print(f"  ⚠️   DROP L2: {stats[Verdict.DROP_L2]}")

src_ip          dst_ip              sp     dp  pr   werdykt
----------------------------------------------------------------------
10.0.0.1        192.168.100.1    50000    443   6   ✅ ALLOW  
10.0.0.2        192.168.100.1    50001    443   6   ✅ ALLOW  
10.0.1.10       192.168.100.1    60000    443   6   ✅ ALLOW  
172.16.0.1      192.168.100.1    50000    443   6   🛑 DROP L1
192.168.1.5     192.168.100.1    12345    443   6   🛑 DROP L1
10.0.0.1        192.168.100.1    50000     22   6   🛑 DROP L1
10.0.0.1        192.168.100.1    50000    443  17   🛑 DROP L1
----------------------------------------------------------------------
Łącznie: 7 pakietów
  ✅  ALLOW  : 3
  🛑  DROP L1: 4
  ⚠️   DROP L2: 0


---
## 5. Inspekcja stanu wewnętrznego firewalla

In [6]:
print("Stan Layer 1 – Bloom Filter")
print(f"  Rozmiar tablicy bitowej : {fw.bloom.size:>10} bitów  (~{fw.bloom.size//8//1024} KB)")
print(f"  Liczba funkcji hash (k) : {fw.bloom.k:>10}")
print(f"  Fill ratio              : {fw.bloom.fill_ratio:>10.2%}")
print()
print("Stan Layer 2 – Cuckoo Hash")
print(f"  Liczba banków           : {fw.cuckoo.num_banks:>10}")
print(f"  Sloty na bank           : {fw.cuckoo.bank_size:>10}")
total = fw.cuckoo.num_banks * fw.cuckoo.bank_size
filled = sum(1 for b in fw.cuckoo._banks for s in b if s is not None)
print(f"  Zajęte sloty            : {filled:>10} / {total}")
print(f"  Load factor             : {fw.cuckoo.load_factor:>10.2%}")

Stan Layer 1 – Bloom Filter
  Rozmiar tablicy bitowej :     147456 bitów  (~18 KB)
  Liczba funkcji hash (k) :         10
  Fill ratio              :      0.03%

Stan Layer 2 – Cuckoo Hash
  Liczba banków           :          3
  Sloty na bank           :      16384
  Zajęte sloty            :          5 / 49152
  Load factor             :      0.01%


---
## 6. Reset i przeładowanie reguł

Jeśli chcesz zmienić whitelist, uruchom tę komórkę — tworzy nową instancję firewalla.

In [7]:
# ── Edytuj tutaj — nowa lista reguł ──────────────────────────────────────────
NEW_RULES = [
    ("10.10.0.1", "192.168.100.1", 55000, 443, 6),
    ("10.10.0.2", "192.168.100.1", 55001, 443, 6),
]
# ─────────────────────────────────────────────────────────────────────────────

fw = AegisPipeline()   # reset
fw.load_rules([
    FiveTuple(ip_to_int(r[0]), ip_to_int(r[1]), r[2], r[3], r[4])
    for r in NEW_RULES
])

print(f"Firewall zresetowany. Załadowano {len(NEW_RULES)} nowych reguł.")
print(f"  Bloom fill ratio  : {fw.bloom.fill_ratio:.2%}")
print(f"  Cuckoo load factor: {fw.cuckoo.load_factor:.2%}")

Firewall zresetowany. Załadowano 2 nowych reguł.
  Bloom fill ratio  : 0.01%
  Cuckoo load factor: 0.00%
